# CRISP-DM: MIR tanah → NPK dengan learning curve hingga 1.000 sampel

Notebook audit untuk artefak yang sama dengan dashboard. Jalankan `python scripts/prepare_data.py` dan `python scripts/run_benchmark.py` sebelum notebook ini.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from npk_spectra import DEFAULT_TARGETS, load_processed_dataset
dataset = load_processed_dataset(PROJECT_ROOT)
predictions = pd.read_csv(PROJECT_ROOT / 'artifacts/predictions.csv')
metrics = pd.read_csv(PROJECT_ROOT / 'artifacts/fold_metrics.csv')
summary = pd.read_csv(PROJECT_ROOT / 'artifacts/summary.csv')
attention_path = PROJECT_ROOT / 'artifacts/attention_summary.csv'
attention_summary = pd.read_csv(attention_path) if attention_path.exists() else pd.DataFrame()
zoo_path = PROJECT_ROOT / 'artifacts/zoo_summary.csv'
zoo_summary = pd.read_csv(zoo_path) if zoo_path.exists() else pd.DataFrame()
dataset.n_samples, dataset.n_features

## 1. Business understanding
Tujuan PoC adalah mengukur apakah informasi spektral memberi sinyal prediktif di atas baseline median pada learning curve 60–1.000 sampel. Ini bukan validasi transfer ke instrumen lokal.

## 2. Data understanding
Satu baris adalah satu pasangan MIR–hasil lab. Definisi target membawa metode dan unit, bukan hanya nama unsur.

In [ ]:
pd.DataFrame([target.to_dict() for target in DEFAULT_TARGETS])

In [ ]:
dataset.metadata[['N', 'P', 'K']].describe().T

In [ ]:
fig = go.Figure()
for row in range(min(20, dataset.n_samples)):
    fig.add_scatter(x=dataset.grid, y=dataset.spectra[row], mode='lines', opacity=.25, showlegend=False)
fig.update_layout(title='20 spektrum MIR pertama', xaxis_title='Wavenumber (cm⁻¹)', yaxis_title='Absorbance', xaxis_autorange='reversed')
fig.show()

## 3–4. Data preparation dan modeling
SNV/turunan Savitzky–Golay serta pemilihan PLS/PCA–Ridge dilakukan di dalam nested CV. Alternatif neural memakai SNV dan patch transformer 108.803 parameter dengan head gabungan N/P/K. Transformasi target `log1p` digunakan oleh kedua family model.

In [ ]:
summary[['target', 'budget', 'rmse_median', 'r2_median', 'rpiq_median', 'rmse_improvement_pct_median']]

## 5. Evaluation
Learning curve harus dibaca bersama interval antar-fold dan baseline. Nilai R² negatif berarti model lebih buruk daripada prediktor konstan pada fold tersebut.

In [ ]:
if not attention_summary.empty or not zoo_summary.empty:
    zoo = zoo_summary.rename(columns={'family': 'model'}) if not zoo_summary.empty else pd.DataFrame()
    comparison = pd.concat([summary.assign(model='Klasik'), attention_summary.assign(model='Self-attention'), zoo], ignore_index=True)
    display(comparison[comparison.budget == 1000][['model', 'target', 'rmse_median', 'r2_median', 'rpiq_median']])

In [ ]:
px.line(summary, x='budget', y='r2_median', color='target', markers=True, title='Learning curve R² median').show()
px.line(summary, x='budget', y='rmse_improvement_pct_median', color='target', markers=True, title='Perbaikan RMSE terhadap baseline (%)').show()

In [ ]:
oof = predictions[predictions.budget == 1000].groupby(['target', 'sample_id'], as_index=False).agg(observed=('observed','first'), predicted=('predicted','mean'))
px.scatter(oof, x='observed', y='predicted', facet_col='target', color='target', trendline=None, title='Observed vs out-of-fold prediction @1000').show()

## 6. Deployment decision
PoC berhenti pada demo OSSL. Langkah berikutnya adalah menyamakan metode lab, membuat loader Shimadzu, dan memakai grouped CV berdasarkan lokasi/batch sebelum prediksi lokal dipertimbangkan.